# Scheduler Patterns Examples

This notebook summarizes common Databricks scheduling and orchestration patterns:

- Databricks native scheduler
- external schedulers such as Airflow and Azure Data Factory
- event-driven orchestration
- scheduler selection tradeoffs
- simple orchestration design patterns

## Pattern 1: Databricks native scheduling

Use Databricks Jobs and Workflows when the orchestration stays mostly inside Databricks.

Good fit:

- notebook or Python task pipelines inside Databricks
- scheduled bronze, silver, gold processing
- teams that do not need a broader enterprise orchestrator

In [ ]:
native_scheduler_pattern = {
    "scheduler": "databricks_jobs",
    "trigger": "cron",
    "tasks": ["ingest", "validate", "publish"],
    "best_for": "Databricks-centric workflows"
}

native_scheduler_pattern

## Pattern 2: Airflow triggers Databricks

Use Airflow when Databricks is one task inside a larger DAG with upstream and downstream dependencies.

In [ ]:
airflow_example = """
from airflow.providers.databricks.operators.databricks import DatabricksRunNowOperator

run_orders_job = DatabricksRunNowOperator(
    task_id=\"run_orders_job\",
    databricks_conn_id=\"databricks_default\",
    job_id=12345,
    job_parameters={\"run_date\": \"{{ ds }}\"},
)
"""

print(airflow_example)

## Pattern 3: Azure Data Factory triggers Databricks

Use ADF when Databricks is part of a broader Azure integration workflow with storage, SQL, and pipeline activities.

In [ ]:
adf_activity_example = {
    "name": "Run_Databricks_Orders_Notebook",
    "type": "DatabricksNotebook",
    "typeProperties": {
        "notebookPath": "/Workspace/Shared/notebooks/ingest_orders",
        "baseParameters": {
            "run_date": "@{pipeline().TriggerTime}",
            "run_mode": "incremental"
        }
    }
}

adf_activity_example

## Pattern 4: Event-driven orchestration

Use event-driven orchestration when file arrivals, messages, or upstream application events should trigger Databricks instead of a fixed cron schedule.

In [ ]:
event_driven_pattern = {
    "trigger_source": "file_arrival_or_message",
    "trigger_target": "databricks_jobs_api",
    "best_for": "near_real_time_or_upstream_driven_processing"
}

event_driven_pattern

## Selection guide

- Use Databricks native scheduling when Databricks is the processing center
- Use Airflow when you need complex cross-system DAG orchestration
- Use ADF when you want Azure-native integration workflows
- Use event-driven patterns when cron-based scheduling is not enough